# 03 — Robust hourly allocation and bid-aware replay

**Goal:** solve the central-cost, upper-cost, and volume-policy MILPs under common
constraints, then replay decisions in strict timestamp order.


## Paper protocol implemented

For each opportunity, the policy bids before outcomes are revealed. A win requires the bid
to clear both paying price and slot floor; the win is charged `payprice`, and click is
counted only for a win. Unlike the paper, this implementation forbids even one-auction
overspend. Because the mirror is impression-supported, the replay cannot evaluate
historically lost auctions.


In [ ]:
from pathlib import Path
import os

# Run correctly whether Jupyter starts in the project root or in notebooks/.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
ARTIFACT_ROOT = Path(os.getenv("IPINYOU_ARTIFACT_ROOT", PROJECT_ROOT / "artifacts"))
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")
print(f"Artifact root: {ARTIFACT_ROOT}")


In [ ]:
from pathlib import Path
from itertools import product
import json
import os

import joblib
import numpy as np
import pandas as pd
from scipy import optimize, sparse

try:
    from IPython.display import Markdown, display
except ImportError:
    class Markdown(str):
        pass
    def display(*objects):
        for obj in objects:
            print(obj)

NOTEBOOK_SCHEMA_VERSION = "2.2"
PRIMARY_BUDGET_FRACTION = 0.50
BUDGET_FRACTIONS = [0.20, 0.35, 0.50, 0.65, 0.80]
PAPER_BUDGET_FRACTIONS = [1 / 32, 1 / 8, 1 / 2]
RUN_PAPER_BUDGET_SENSITIVITY = True
SOLVER_TIME_LIMIT_SEC = 30
SOLVER_MIP_REL_GAP = 1e-7
MAX_ACCEPTABLE_REPORTED_MIP_GAP = 1e-3
REQUIRE_BID_TO_CLEAR = True
REQUIRE_FLOOR_TO_CLEAR = True
GROUP_COLS = ["hour_key", "adexchange_key", "pctr_bucket", "cost_bucket"]

MODEL_DIR = ARTIFACT_ROOT / "02_models_and_scores"
model_manifest_path = MODEL_DIR / "model_manifest.json"
if not model_manifest_path.exists():
    raise FileNotFoundError("Run 02_model_tuning_and_calibration.ipynb first.")
model_manifest = json.loads(model_manifest_path.read_text(encoding="utf-8"))
ELIGIBLE_ADVERTISERS = [int(x) for x in model_manifest["eligible_advertisers"]]

def calendar_hour_slots(holdout_frame):
    start = holdout_frame.event_time.min().floor("h")
    end = holdout_frame.event_time.max().floor("h")
    return list(pd.date_range(start, end, freq="h"))


## 9. Hourly integer allocation

For the current evaluation hour, the optimizer solves

\[
\max_x\sum_g v_gx_g
\]

subject to

\[
\sum_g c_gx_g\le B_h,\qquad 0\le x_g\le N_g,\qquad x_g\in\mathbb Z_+.
\]

The upper-cost policy uses \(u_g\) rather than \(c_g\). Equivalently, it is the
box-robust counterpart for the declared coefficient set
\(\tilde c_g\in[c_g,u_g]\). Because the upper bounds are empirically calibrated,
their validation/test coverage is shown; no campaign-level chance guarantee is inferred.

Every solve is current-hour only. Later hours are reoptimized using remaining realized budget.
The sum of all reoptimized quota costs is therefore a diagnostic—not a committed multi-period
plan—and is named accordingly in the result table.


In [ ]:
def prepare_hour_weights(planning, cost_column):
    weighted = planning.copy()
    weighted["forecast_spend"] = weighted[cost_column] * weighted["forecast_capacity"]
    return weighted.groupby("hour_key", observed=True).forecast_spend.sum().astype(float).to_dict()


def solve_hour_ilp(
    planning,
    hour_key,
    budget,
    cost_column="central_cost",
    value_column="expected_ctr",
    time_limit=SOLVER_TIME_LIMIT_SEC,
):
    hour_plan = planning.loc[
        planning.hour_key.astype(str) == str(hour_key)
    ].reset_index(drop=True).copy()
    if hour_plan.empty or budget <= 0:
        return {
            "status": "no_inventory", "message": "No planning groups or no budget",
            "budget": float(budget), "cost_column": cost_column,
            "value_column": value_column, "mip_gap": 0.0, "allocation": None,
            "quota_cost": 0.0, "quota_impressions": 0, "quota_value": 0.0,
            "feasibility_violation": 0.0,
        }
    n_groups = len(hour_plan)
    objective = -hour_plan[value_column].astype(float).to_numpy()
    lower = np.zeros(n_groups)
    upper = hour_plan.forecast_capacity.astype(float).to_numpy()
    integrality = np.ones(n_groups, dtype=int)
    costs = hour_plan[cost_column].astype(float).to_numpy()
    constraint = optimize.LinearConstraint(
        sparse.csr_matrix(costs.reshape(1, -1)),
        np.array([-np.inf]), np.array([float(budget)]),
    )
    result = optimize.milp(
        c=objective,
        integrality=integrality,
        bounds=optimize.Bounds(lower, upper),
        constraints=constraint,
        options={"time_limit": time_limit, "mip_rel_gap": SOLVER_MIP_REL_GAP},
    )
    status_map = {0: "optimal", 1: "limit_reached", 2: "infeasible", 3: "unbounded", 4: "other_failure"}
    status = status_map.get(result.status, f"status_{result.status}")
    output = {
        "status": status, "message": result.message, "budget": float(budget),
        "cost_column": cost_column, "value_column": value_column,
        "mip_gap": getattr(result, "mip_gap", np.nan), "allocation": None,
        "quota_cost": np.nan, "quota_impressions": 0, "quota_value": np.nan,
        "feasibility_violation": np.nan,
    }
    if status != "optimal" or result.x is None:
        return output
    allocation_vector = np.rint(np.clip(result.x, 0, upper)).astype(int)
    quota_cost = float(costs @ allocation_vector)
    violation = max(0.0, quota_cost - float(budget))
    assert np.all(allocation_vector >= 0) and np.all(allocation_vector <= upper + 1e-9)
    assert violation <= 1e-5 * max(1.0, float(budget))
    allocation = hour_plan[
        ["group_id"] + GROUP_COLS
        + ["forecast_capacity", "expected_ctr", "central_cost", "upper_cost"]
    ].copy()
    allocation["quota_impressions"] = allocation_vector
    allocation = allocation[allocation.quota_impressions > 0].copy()
    output.update(
        {
            "allocation": allocation,
            "quota_cost": quota_cost,
            "quota_impressions": int(allocation_vector.sum()),
            "quota_value": float(np.sum(hour_plan[value_column].to_numpy() * allocation_vector)),
            "feasibility_violation": violation,
        }
    )
    return output


def quota_from_plan(plan):
    if plan["allocation"] is None:
        return {}
    return {
        (str(row.hour_key), str(row.adexchange_key), int(row.pctr_bucket), int(row.cost_bucket)):
        int(row.quota_impressions)
        for row in plan["allocation"].itertuples()
    }


## 10. Bid-aware strict future replay

For each logged opportunity, the replay follows this order:

1. Look up its frozen segment and remaining quota.
2. Check the row-level predicted budget using the policy's cost estimate.
3. Submit that estimate as a bid cap.
4. Reveal `payprice` and `slotprice`; the bid must clear both market price and floor.
5. Reveal and count `click` only for won impressions.
6. Update realized budget and reoptimize the next hour.

This rule is intentionally simple: the ML-plus-MILP layer chooses which opportunities are worth
attempting, and the price model provides the bid ceiling. It is more coherent than assuming every
selected auction is automatically purchased, but it is still not claimed to be an optimized
production bid function.


In [ ]:
def rolling_replay(
    planning,
    holdout_frame,
    slot_groups,
    total_budget,
    policy_name,
    cost_column,
    value_column,
):
    remaining_realized_budget = float(total_budget)
    accepted_rows = []
    reoptimized_quota_cost_sum = 0.0
    reoptimized_quota_impressions = 0
    predicted_winning_spend = 0.0
    slot_rows = []
    solver_statuses = []
    max_mip_gap = 0.0
    max_feasibility_violation = 0.0
    bid_attempts_total = 0
    bid_losses_total = 0
    market_price_losses_total = 0
    floor_losses_total = 0

    slots = calendar_hour_slots(holdout_frame)
    hour_weights = prepare_hour_weights(planning, cost_column)
    row_cost_field = "upper_cost_pred" if cost_column == "upper_cost" else "central_cost_pred"

    for slot_index, slot_timestamp in enumerate(slots):
        if remaining_realized_budget <= 0:
            break
        hour = int(slot_timestamp.hour)
        remaining_slots = slots[slot_index:]
        denominator = sum(float(hour_weights.get(str(ts.hour), 0.0)) for ts in remaining_slots)
        hour_weight = float(hour_weights.get(str(hour), 0.0))
        slot_budget = (
            remaining_realized_budget * hour_weight / denominator
            if denominator > 0 and hour_weight > 0 else 0.0
        )
        plan = solve_hour_ilp(
            planning, str(hour), slot_budget,
            cost_column=cost_column, value_column=value_column,
        )
        if plan["status"] != "no_inventory":
            solver_statuses.append(plan["status"])
        if np.isfinite(plan.get("mip_gap", np.nan)):
            max_mip_gap = max(max_mip_gap, float(plan["mip_gap"]))
        if np.isfinite(plan.get("feasibility_violation", np.nan)):
            max_feasibility_violation = max(
                max_feasibility_violation, float(plan["feasibility_violation"])
            )

        if plan["allocation"] is None:
            slot_rows.append(
                {
                    "slot": slot_timestamp, "day": slot_timestamp.date(), "hour": hour,
                    "allocated_budget": slot_budget, "reoptimized_quota_cost": 0.0,
                    "predicted_winning_spend": 0.0, "realized_spend": 0.0,
                    "clicks": 0, "wins": 0, "bid_attempts": 0, "bid_losses": 0,
                    "market_price_losses": 0, "floor_losses": 0,
                    "solver_status": plan["status"],
                }
            )
            continue

        reoptimized_quota_cost_sum += plan["quota_cost"]
        reoptimized_quota_impressions += plan["quota_impressions"]
        quota = quota_from_plan(plan)
        slot_predicted_budget_remaining = float(slot_budget)
        slot_predicted_spend = 0.0
        slot_realized_spend = 0.0
        slot_clicks = 0
        slot_wins = 0
        slot_attempts = 0
        slot_losses = 0
        slot_market_losses = 0
        slot_floor_losses = 0

        slot_data = slot_groups.get(slot_timestamp)
        if slot_data is not None:
            for row in slot_data.itertuples():
                if remaining_realized_budget <= 0:
                    break
                key = (
                    str(row.hour_key), str(row.adexchange_key),
                    int(row.pctr_bucket), int(row.cost_bucket),
                )
                if quota.get(key, 0) <= 0:
                    continue
                decision_cost = float(getattr(row, row_cost_field))
                if not np.isfinite(decision_cost) or decision_cost <= 0:
                    continue
                if decision_cost > slot_predicted_budget_remaining:
                    continue
                if decision_cost > remaining_realized_budget:
                    continue

                # The decision is now frozen. payprice is revealed only to determine the win.
                slot_attempts += 1
                actual_cost = float(row.payprice)
                floor_price = float(row.slotprice)
                clears_market = (
                    decision_cost + 1e-12 >= actual_cost
                ) if REQUIRE_BID_TO_CLEAR else True
                clears_floor = (
                    decision_cost + 1e-12 >= floor_price
                ) if REQUIRE_FLOOR_TO_CLEAR else True
                won = clears_market and clears_floor
                if not won:
                    slot_losses += 1
                    slot_market_losses += int(not clears_market)
                    slot_floor_losses += int(not clears_floor)
                    continue

                quota[key] -= 1
                slot_predicted_budget_remaining -= decision_cost
                slot_predicted_spend += decision_cost
                remaining_realized_budget -= actual_cost
                slot_realized_spend += actual_cost
                slot_clicks += int(row.click)
                slot_wins += 1
                accepted_rows.append(row.Index)
                if REQUIRE_BID_TO_CLEAR:
                    assert actual_cost <= decision_cost + 1e-9
                if REQUIRE_FLOOR_TO_CLEAR:
                    assert floor_price <= decision_cost + 1e-9

        predicted_winning_spend += slot_predicted_spend
        bid_attempts_total += slot_attempts
        bid_losses_total += slot_losses
        market_price_losses_total += slot_market_losses
        floor_losses_total += slot_floor_losses
        slot_rows.append(
            {
                "slot": slot_timestamp, "day": slot_timestamp.date(), "hour": hour,
                "allocated_budget": slot_budget,
                "reoptimized_quota_cost": plan["quota_cost"],
                "predicted_winning_spend": slot_predicted_spend,
                "realized_spend": slot_realized_spend,
                "clicks": slot_clicks, "wins": slot_wins,
                "bid_attempts": slot_attempts, "bid_losses": slot_losses,
                "market_price_losses": slot_market_losses,
                "floor_losses": slot_floor_losses,
                "solver_status": plan["status"],
            }
        )

    slots_frame = pd.DataFrame(slot_rows)
    realized_spend = float(slots_frame.realized_spend.sum()) if len(slots_frame) else 0.0
    clicks = int(holdout_frame.loc[accepted_rows, "click"].sum()) if accepted_rows else 0
    wins = len(accepted_rows)
    daily = slots_frame.groupby("day", as_index=False).agg(
        allocated_budget=("allocated_budget", "sum"),
        predicted_winning_spend=("predicted_winning_spend", "sum"),
        realized_spend=("realized_spend", "sum"),
        clicks=("clicks", "sum"), wins=("wins", "sum"),
        bid_attempts=("bid_attempts", "sum"), bid_losses=("bid_losses", "sum"),
    ) if len(slots_frame) else pd.DataFrame()
    return {
        "policy": policy_name,
        "budget": float(total_budget),
        "reoptimized_quota_cost_sum": reoptimized_quota_cost_sum,
        "predicted_winning_spend": predicted_winning_spend,
        "reoptimized_quota_impressions": reoptimized_quota_impressions,
        "accepted_impressions": wins,
        "bid_attempts": bid_attempts_total,
        "bid_losses": bid_losses_total,
        "market_price_losses": market_price_losses_total,
        "floor_losses": floor_losses_total,
        "auction_win_rate": wins / bid_attempts_total if bid_attempts_total else np.nan,
        "realized_spend": realized_spend,
        "budget_utilization": realized_spend / total_budget if total_budget else np.nan,
        "budget_overrun_pct": max(0.0, realized_spend - total_budget) / total_budget if total_budget else np.nan,
        "realized_clicks": clicks,
        "realized_ctr": clicks / wins if wins else np.nan,
        "realized_cpc_rmb": (realized_spend / 1000) / clicks if clicks else np.nan,
        "clicks_per_1000_rmb": clicks / (realized_spend / 1e6) if realized_spend > 0 else np.nan,
        "accepted_rows": accepted_rows,
        "daily": daily,
        "slots": slots_frame,
        "all_solver_optimal": all(status == "optimal" for status in solver_statuses) if solver_statuses else True,
        "max_mip_gap": max_mip_gap,
        "max_feasibility_violation": max_feasibility_violation,
    }


## Execute policy scenarios

Train-projected budgets are primary. The paper's 1/32, 1/8, and 1/2 test-cost fractions
are saved separately as a benchmark sensitivity and never drive tuning or headline claims.


In [ ]:
POLICY_DIR = ARTIFACT_ROOT / "03_policy_replay"
POLICY_DIR.mkdir(parents=True, exist_ok=True)

policy_specs = [
    ("CTR central-cost", "central_cost", "expected_ctr"),
    ("CTR upper-cost", "upper_cost", "expected_ctr"),
    ("Volume benchmark", "central_cost", "unit_value"),
]

def replay_result_row(advertiser, fraction, budget_reference, replay):
    return {
        "advertiser": advertiser,
        "budget_fraction": fraction,
        "budget_reference": budget_reference,
        "policy": replay["policy"],
        "scenario_budget_native": replay["budget"],
        "scenario_budget_rmb": replay["budget"] / 1000,
        "reoptimized_quota_cost_sum_native": replay["reoptimized_quota_cost_sum"],
        "predicted_winning_spend_native": replay["predicted_winning_spend"],
        "reoptimized_quota_impressions": replay["reoptimized_quota_impressions"],
        "accepted_impressions": replay["accepted_impressions"],
        "bid_attempts": replay["bid_attempts"],
        "market_price_losses": replay["market_price_losses"],
        "floor_losses": replay["floor_losses"],
        "auction_win_rate": replay["auction_win_rate"],
        "realized_spend_native": replay["realized_spend"],
        "realized_spend_rmb": replay["realized_spend"] / 1000,
        "budget_utilization": replay["budget_utilization"],
        "budget_overrun_pct": replay["budget_overrun_pct"],
        "realized_clicks": replay["realized_clicks"],
        "realized_ctr": replay["realized_ctr"],
        "realized_cpc_rmb": replay["realized_cpc_rmb"],
        "clicks_per_1000_rmb": replay["clicks_per_1000_rmb"],
        "all_solver_optimal": replay["all_solver_optimal"],
        "max_mip_gap": replay["max_mip_gap"],
        "max_feasibility_violation": replay["max_feasibility_violation"],
    }

primary_rows, paper_rows = [], []
for run_number, advertiser in enumerate(ELIGIBLE_ADVERTISERS, 1):
    print(f"[{run_number}/{len(ELIGIBLE_ADVERTISERS)}] replay advertiser {advertiser}")
    planning = pd.read_parquet(MODEL_DIR / f"advertiser_{advertiser}_planning.parquet")
    test_scored = pd.read_parquet(MODEL_DIR / f"advertiser_{advertiser}_test_scored.parquet")
    slot_groups = {
        timestamp: group.sort_values("event_time", kind="mergesort")
        for timestamp, group in test_scored.groupby("slot_key", sort=True)
    }
    full_spend_reference = float(
        model_manifest["full_spend_reference_native"][str(advertiser)]
    )
    primary_replays = {}
    for fraction in BUDGET_FRACTIONS:
        budget = fraction * full_spend_reference
        for name, cost_column, value_column in policy_specs:
            replay = rolling_replay(
                planning, test_scored, slot_groups, budget, name, cost_column, value_column
            )
            primary_replays[(fraction, name)] = replay
            primary_rows.append(
                replay_result_row(advertiser, fraction, "train_projected", replay)
            )
    joblib.dump(
        primary_replays,
        POLICY_DIR / f"advertiser_{advertiser}_primary_replays.joblib",
        compress=3,
    )

    # Paper-aligned budget fractions are reported separately because their denominator uses
    # strict-test realized cost, information unavailable at deployment time.
    if RUN_PAPER_BUDGET_SENSITIVITY:
        test_cost_reference = float(test_scored.payprice.sum())
        for fraction in PAPER_BUDGET_FRACTIONS:
            budget = fraction * test_cost_reference
            for name, cost_column, value_column in policy_specs:
                replay = rolling_replay(
                    planning, test_scored, slot_groups, budget, name, cost_column, value_column
                )
                paper_rows.append(
                    replay_result_row(advertiser, fraction, "strict_test_realized_cost", replay)
                )

results = pd.DataFrame(primary_rows)
results.to_csv(POLICY_DIR / "primary_results.csv", index=False)
paper_budget_results = pd.DataFrame(paper_rows, columns=results.columns)
paper_budget_results.to_csv(POLICY_DIR / "paper_budget_sensitivity.csv", index=False)
display(results.sort_values(["advertiser", "budget_fraction", "policy"]))
if len(paper_budget_results):
    display(Markdown("### Paper-budget sensitivity (diagnostic, not the primary result)"))
    display(paper_budget_results.sort_values(["advertiser", "budget_fraction", "policy"]))
print(f"Replay artifacts written to {POLICY_DIR}")
